# 🌍 Global Food Security & Climate Resilience
### Data Preparation & Integration

---

## 📖 Project Overview

This notebook prepares, cleans, standardizes, and integrates multiple real-world datasets into a single analysis-ready dataset for evaluating global food security and climate resilience across **167 countries** between **2010 and 2023**.

The resulting dataset serves as the foundation for the project's exploratory analysis, analytical visualizations, and interactive Streamlit dashboard.

---

## 📂 Data Sources

| Dataset | Indicator | Source |
|:---------|:----------|:-------|
| Food Balance Sheets | Food Supply (Calories per Capita) | FAOSTAT – https://www.fao.org/faostat/en/#data/FBS |
| Crops and Livestock Products | Import Quantity (Food Import Dependency) | FAOSTAT – https://www.fao.org/faostat/en/#data/ET |
| GDP per Capita | Current US$ | World Bank – https://data.worldbank.org/indicator/NY.GDP.PCAP.CD |
| Population | Total Population | World Bank – https://data.worldbank.org/indicator/SP.POP.TOTL |
| Agricultural Land | % of Land Area | World Bank – https://data.worldbank.org/indicator/AG.LND.AGRI.ZS |

> **Note:** Country names, reporting years, and variable formats are standardized before integrating the datasets into a unified analytical dataset.

---

## ⚙️ Data Preparation Workflow

- Import datasets
- Assess data quality
- Clean and standardize variables
- Handle missing values
- Integrate datasets
- Engineer analytical features
- Validate the final dataset
- Export the analysis-ready dataset

---

> **Output:** A clean, validated, and integrated dataset ready for exploratory analysis and visualization.


# PHASE 1 — Import Libraries & Project Setup

In [1]:
# ==========================================================
# Data Visualization Final Project
# Notebook 1 : Data Preparation
# Phase 1 - Import Libraries & Project Setup
# ==========================================================

import pandas as pd
import numpy as np

from pathlib import Path

!pip install pycountry_convert --quiet

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==========================================================
# Project Directory
# ==========================================================

PROJECT_DIR = Path.cwd()

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

OUTPUT_DIR = PROJECT_DIR / "outputs"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project Directory:")
print(PROJECT_DIR)

print("\nRaw Data:")
print(RAW_DATA_DIR)

print("\nProcessed Data:")
print(PROCESSED_DATA_DIR)

print("\nOutputs:")
print(OUTPUT_DIR)

Project Directory:
D:\My Folder - Anupriya\UE\UE Semester 2\Data Visualization\DataViz_Final_Project

Raw Data:
D:\My Folder - Anupriya\UE\UE Semester 2\Data Visualization\DataViz_Final_Project\data\raw

Processed Data:
D:\My Folder - Anupriya\UE\UE Semester 2\Data Visualization\DataViz_Final_Project\data\processed

Outputs:
D:\My Folder - Anupriya\UE\UE Semester 2\Data Visualization\DataViz_Final_Project\outputs


In [3]:
# ==========================================================
# Verify Folder Structure
# ==========================================================

print("Contents of Project Directory\n")

for item in PROJECT_DIR.iterdir():
    print(item.name)

Contents of Project Directory

.gitignore.txt
.ipynb_checkpoints
.streamlit
01_Data_Preparation.ipynb
02_Analytical_Questions_v4.ipynb
data
others
outputs
pages
requirements.txt


# PHASE 2 — Load Raw Data

In [4]:
# ==========================================================
# Dataset Paths
# ==========================================================

food_file = RAW_DATA_DIR / "FoodBalance.csv"

temperature_file = RAW_DATA_DIR / "Temperature.csv"

gdp_file = RAW_DATA_DIR / "GDP.csv"

population_file = RAW_DATA_DIR / "Population.csv"

land_file = RAW_DATA_DIR / "AgriculturalLand.csv"

In [5]:
# ==========================================================
# Load Food Balance
# ==========================================================

food_df = pd.read_csv(food_file)

print(food_df.shape)

food_df.head()

(4820497, 14)


,Area Code,Area Code (M49),Area,Item Code,Item Code (FBS),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
0,2,'004,Afghanistan,2501,'S2501,Population,511,Total Population - Both sexes,2010,2010,1000 No,28284.09,X,NaN
1,3,'008,Albania,2501,'S2501,Population,511,Total Population - Both sexes,2010,2010,1000 No,2928.72,X,NaN
2,2,'004,Afghanistan,2501,'S2501,Population,511,Total Population - Both sexes,2011,2011,1000 No,29347.71,X,NaN
3,3,'008,Albania,2501,'S2501,Population,511,Total Population - Both sexes,2011,2011,1000 No,2911.50,X,NaN
4,2,'004,Afghanistan,2501,'S2501,Population,511,Total Population - Both sexes,2012,2012,1000 No,30560.03,X,NaN


In [6]:
# ==========================================================
# Load Temperature
# ==========================================================

temperature_df = pd.read_csv(temperature_file)

print(temperature_df.shape)

temperature_df.head()

(590512, 12)


,Area Code,Area Code (M49),Area,Months Code,Months,Element Code,Element,Year Code,Year,Unit,Value,Flag
0,2,'004,Afghanistan,7001,January,7271,Temperature change,1961,1961,°C,0.77,E
1,2,'004,Afghanistan,7001,January,7271,Temperature change,1962,1962,°C,0.03,E
2,2,'004,Afghanistan,7001,January,7271,Temperature change,1963,1963,°C,2.71,E
3,2,'004,Afghanistan,7001,January,7271,Temperature change,1964,1964,°C,-5.26,E
4,2,'004,Afghanistan,7001,January,7271,Temperature change,1965,1965,°C,1.85,E


In [7]:
# ==========================================================
# Helper Function for World Bank Datasets
# ==========================================================

def load_world_bank(path):

    df = pd.read_csv(
        path,
        skiprows=4
    )

    # remove empty last column if present
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    return df

In [8]:
# ==========================================================
# Load World Bank Files
# ==========================================================

gdp_df = load_world_bank(gdp_file)

population_df = load_world_bank(population_file)

land_df = load_world_bank(land_file)

print("GDP:", gdp_df.shape)
print("Population:", population_df.shape)
print("Agricultural Land:", land_df.shape)

GDP: (266, 69)
Population: (265, 70)
Agricultural Land: (265, 70)


# PHASE 3 — Standardize World Bank Datasets

In [9]:
# ==========================================================
# Standardize World Bank Dataset
# ==========================================================

def standardize_worldbank(df, value_name):

    years = [str(y) for y in range(1960, 2026)]

    available_years = [y for y in years if y in df.columns]

    df_long = df.melt(
        id_vars=["Country Name"],
        value_vars=available_years,
        var_name="Year",
        value_name=value_name
    )

    df_long.rename(
        columns={"Country Name": "Country"},
        inplace=True
    )

    df_long["Year"] = df_long["Year"].astype(int)

    return df_long

In [10]:
# ==========================================================
# Standardize All World Bank Files
# ==========================================================

gdp_long = standardize_worldbank(
    gdp_df,
    "GDP_per_Capita"
)

population_long = standardize_worldbank(
    population_df,
    "Population"
)

land_long = standardize_worldbank(
    land_df,
    "Agricultural_Land_Percent"
)

In [11]:
# ==========================================================
# Verify Standardized Datasets
# ==========================================================

datasets = {
    "GDP": gdp_long,
    "Population": population_long,
    "Agricultural Land": land_long
}

for name, df in datasets.items():

    print("="*80)
    print(name)
    print("="*80)

    print(df.shape)

    display(df.head())

    print(df.dtypes)

GDP
(17290, 3)


,Country,Year,GDP_per_Capita
0,Aruba,1960,NaN
1,Africa Eastern and Southern,1960,186.09
2,Afghanistan,1960,NaN
3,Africa Western and Central,1960,121.94
4,Angola,1960,NaN


Country            object
Year                int64
GDP_per_Capita    float64
dtype: object
Population
(17490, 3)


,Country,Year,Population
0,Aruba,1960,54922.00
1,Africa Eastern and Southern,1960,130075728.00
2,Afghanistan,1960,9035043.00
3,Africa Western and Central,1960,97630925.00
4,Angola,1960,5231654.00


Country        object
Year            int64
Population    float64
dtype: object
Agricultural Land
(17490, 3)


,Country,Year,Agricultural_Land_Percent
0,Aruba,1960,NaN
1,Africa Eastern and Southern,1960,NaN
2,Afghanistan,1960,NaN
3,Africa Western and Central,1960,NaN
4,Angola,1960,NaN


Country                       object
Year                           int64
Agricultural_Land_Percent    float64
dtype: object


In [12]:
# ==========================================================
# Save Standardized Files (Optional Backup)
# ==========================================================

gdp_long.to_csv(
    PROCESSED_DATA_DIR / "GDP_Standardized.csv",
    index=False
)

population_long.to_csv(
    PROCESSED_DATA_DIR / "Population_Standardized.csv",
    index=False
)

land_long.to_csv(
    PROCESSED_DATA_DIR / "AgriculturalLand_Standardized.csv",
    index=False
)

print("Standardized datasets saved successfully.")

Standardized datasets saved successfully.


# Phase 4 – Explore & Understand the Raw Data

In [13]:
# ==========================================================
# Phase 4.1 - Food Balance Overview
# ==========================================================

print("="*80)
print("FOOD BALANCE DATASET")
print("="*80)

print("Shape:", food_df.shape)

display(food_df.head())

print("\nColumns")
display(food_df.columns.tolist())

print("\nData Types")
print(food_df.dtypes)

print("\nMissing Values")
display(food_df.isnull().sum())

FOOD BALANCE DATASET
Shape: (4820497, 14)


,Area Code,Area Code (M49),Area,Item Code,Item Code (FBS),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
0,2,'004,Afghanistan,2501,'S2501,Population,511,Total Population - Both sexes,2010,2010,1000 No,28284.09,X,NaN
1,3,'008,Albania,2501,'S2501,Population,511,Total Population - Both sexes,2010,2010,1000 No,2928.72,X,NaN
2,2,'004,Afghanistan,2501,'S2501,Population,511,Total Population - Both sexes,2011,2011,1000 No,29347.71,X,NaN
3,3,'008,Albania,2501,'S2501,Population,511,Total Population - Both sexes,2011,2011,1000 No,2911.50,X,NaN
4,2,'004,Afghanistan,2501,'S2501,Population,511,Total Population - Both sexes,2012,2012,1000 No,30560.03,X,NaN



Columns


['Area Code',
 'Area Code (M49)',
 'Area',
 'Item Code',
 'Item Code (FBS)',
 'Item',
 'Element Code',
 'Element',
 'Year Code',
 'Year',
 'Unit',
 'Value',
 'Flag',
 'Note']


Data Types
Area Code            int64
Area Code (M49)     object
Area                object
Item Code            int64
Item Code (FBS)     object
Item                object
Element Code         int64
Element             object
Year Code            int64
Year                 int64
Unit                object
Value              float64
Flag                object
Note               float64
dtype: object

Missing Values


Area Code                0
Area Code (M49)          0
Area                     0
Item Code                0
Item Code (FBS)          0
Item                     0
Element Code             0
Element                  0
Year Code                0
Year                     0
Unit                     0
Value                    0
Flag                     0
Note               4820497
dtype: int64

In [14]:
# ==========================================================
# Phase 4.2 - Temperature Overview
# ==========================================================

print("="*80)
print("TEMPERATURE DATASET")
print("="*80)

print("Shape:", temperature_df.shape)

display(temperature_df.head())

print("\nColumns")
display(temperature_df.columns.tolist())

print("\nData Types")
print(temperature_df.dtypes)

print("\nMissing Values")
display(temperature_df.isnull().sum())

TEMPERATURE DATASET
Shape: (590512, 12)


,Area Code,Area Code (M49),Area,Months Code,Months,Element Code,Element,Year Code,Year,Unit,Value,Flag
0,2,'004,Afghanistan,7001,January,7271,Temperature change,1961,1961,°C,0.77,E
1,2,'004,Afghanistan,7001,January,7271,Temperature change,1962,1962,°C,0.03,E
2,2,'004,Afghanistan,7001,January,7271,Temperature change,1963,1963,°C,2.71,E
3,2,'004,Afghanistan,7001,January,7271,Temperature change,1964,1964,°C,-5.26,E
4,2,'004,Afghanistan,7001,January,7271,Temperature change,1965,1965,°C,1.85,E



Columns


['Area Code',
 'Area Code (M49)',
 'Area',
 'Months Code',
 'Months',
 'Element Code',
 'Element',
 'Year Code',
 'Year',
 'Unit',
 'Value',
 'Flag']


Data Types
Area Code            int64
Area Code (M49)     object
Area                object
Months Code          int64
Months              object
Element Code         int64
Element             object
Year Code            int64
Year                 int64
Unit                object
Value              float64
Flag                object
dtype: object

Missing Values


Area Code              0
Area Code (M49)        0
Area                   0
Months Code            0
Months                 0
Element Code           0
Element                0
Year Code              0
Year                   0
Unit                   0
Value              22333
Flag                   0
dtype: int64

In [15]:
# ==========================================================
# Phase 4.3 - World Bank Datasets
# ==========================================================

datasets = {
    "GDP": gdp_long,
    "Population": population_long,
    "Agricultural Land": land_long
}

for name, df in datasets.items():

    print("="*80)
    print(name)
    print("="*80)

    print(df.shape)

    display(df.head())

    print(df.dtypes)

    print("\nMissing Values")
    display(df.isnull().sum())

GDP
(17290, 3)


,Country,Year,GDP_per_Capita
0,Aruba,1960,NaN
1,Africa Eastern and Southern,1960,186.09
2,Afghanistan,1960,NaN
3,Africa Western and Central,1960,121.94
4,Angola,1960,NaN


Country            object
Year                int64
GDP_per_Capita    float64
dtype: object

Missing Values


Country              0
Year                 0
GDP_per_Capita    2728
dtype: int64

Population
(17490, 3)


...

Country        0
Year           0
Population    96
dtype: int64

Agricultural Land
(17490, 3)


,Country,Year,Agricultural_Land_Percent
0,Aruba,1960,NaN
1,Africa Eastern and Southern,1960,NaN
2,Afghanistan,1960,NaN
3,Africa Western and Central,1960,NaN
4,Angola,1960,NaN


Country                       object
Year                           int64
Agricultural_Land_Percent    float64
dtype: object

Missing Values


Country                         0
Year                            0
Agricultural_Land_Percent    2408
dtype: int64

# Phase 5 – Keep Only Relevant Variables

In [16]:
# ==========================================================
# Phase 5.1 - Keep Required Food Elements
# ==========================================================

required_elements = [

    "Production",

    "Import quantity",

    "Export quantity",

    "Domestic supply quantity",

    "Food supply quantity (kg/capita/yr)",

    "Food supply (kcal/capita/day)"
]

food_filtered = food_df[
    food_df["Element"].isin(required_elements)
].copy()

print("Original Shape :", food_df.shape)
print("Filtered Shape :", food_filtered.shape)

print("\nElement Counts")
display(food_filtered["Element"].value_counts())

Original Shape : (4820497, 14)
Filtered Shape : (1761484, 14)

Element Counts


Element
Domestic supply quantity               329755
Import quantity                        318901
Food supply (kcal/capita/day)          310263
Food supply quantity (kg/capita/yr)    308034
Export quantity                        268537
Production                             225994
Name: count, dtype: int64

In [17]:
# ==========================================================
# Phase 5.2 - Keep Annual Temperature Change
# ==========================================================

temperature_filtered = temperature_df[

    (temperature_df["Months"] == "Meteorological year") &

    (temperature_df["Element"] == "Temperature change")

].copy()

print("Original Shape :", temperature_df.shape)
print("Filtered Shape :", temperature_filtered.shape)

display(temperature_filtered.head())

Original Shape : (590512, 12)
Filtered Shape : (17368, 12)


,Area Code,Area Code (M49),Area,Months Code,Months,Element Code,Element,Year Code,Year,Unit,Value,Flag
2080,2,'004,Afghanistan,7020,Meteorological year,7271,Temperature change,1961,1961,°C,-0.10,E
2081,2,'004,Afghanistan,7020,Meteorological year,7271,Temperature change,1962,1962,°C,-0.15,E
2082,2,'004,Afghanistan,7020,Meteorological year,7271,Temperature change,1963,1963,°C,0.86,E
2083,2,'004,Afghanistan,7020,Meteorological year,7271,Temperature change,1964,1964,°C,-0.76,E
2084,2,'004,Afghanistan,7020,Meteorological year,7271,Temperature change,1965,1965,°C,-0.23,E


# Phase 6 – Reduce Columns

In [18]:
# ==========================================================
# Phase 6.1 - Food Columns
# ==========================================================

food_filtered = food_filtered[

    [

        "Area",

        "Year",

        "Item",

        "Element",

        "Unit",

        "Value"

    ]

].copy()

food_filtered.head()

,Area,Year,Item,Element,Unit,Value
107,Afghanistan,2010,Grand Total,Food supply (kcal/capita/day),kcal/cap/d,2200.21
128,Afghanistan,2011,Grand Total,Food supply (kcal/capita/day),kcal/cap/d,2171.86
134,Albania,2010,Grand Total,Food supply (kcal/capita/day),kcal/cap/d,3237.43
137,Bahrain,2019,Grand Total,Food supply (kcal/capita/day),kcal/cap/d,3489.08
138,Afghanistan,2012,Grand Total,Food supply (kcal/capita/day),kcal/cap/d,2165.88


In [19]:
# ==========================================================
# Phase 6.2 - Temperature Columns
# ==========================================================

temperature_filtered = temperature_filtered[

    [

        "Area",

        "Year",

        "Value"

    ]

].copy()

temperature_filtered.columns = [

    "Country",

    "Year",

    "Temperature_Change"

]

temperature_filtered.head()

,Country,Year,Temperature_Change
2080,Afghanistan,1961,-0.10
2081,Afghanistan,1962,-0.15
2082,Afghanistan,1963,0.86
2083,Afghanistan,1964,-0.76
2084,Afghanistan,1965,-0.23


# Phase 7 – Data Quality Assessment

In [20]:
# ==========================================================
# Phase 7.1 - Missing Values
# ==========================================================

datasets = {

    "Food": food_filtered,

    "Temperature": temperature_filtered,

    "GDP": gdp_long,

    "Population": population_long,

    "Land": land_long

}

for name, df in datasets.items():

    print("="*60)
    print(name)
    print("="*60)

    print(df.isnull().sum())

Food
Area       0
Year       0
Item       0
Element    0
Unit       0
Value      0
dtype: int64
Temperature
Country                 0
Year                    0
Temperature_Change    580
dtype: int64
GDP
Country              0
Year                 0
GDP_per_Capita    2728
dtype: int64
Population
Country        0
Year           0
Population    96
dtype: int64
Land
Country                         0
Year                            0
Agricultural_Land_Percent    2408
dtype: int64


In [21]:
# ==========================================================
# Phase 7.2 - Duplicate Check
# ==========================================================

print("Food duplicates:",
      food_filtered.duplicated().sum())

print("Temperature duplicates:",
      temperature_filtered.duplicated().sum())

print("GDP duplicates:",
      gdp_long.duplicated().sum())

print("Population duplicates:",
      population_long.duplicated().sum())

print("Land duplicates:",
      land_long.duplicated().sum())

Food duplicates: 37228
Temperature duplicates: 0
GDP duplicates: 0
Population duplicates: 0
Land duplicates: 0


In [22]:
# ==========================================================
# Phase 7.3 - Remove Duplicates
# ==========================================================

food_filtered = food_filtered.drop_duplicates()

temperature_filtered = temperature_filtered.drop_duplicates()

gdp_long = gdp_long.drop_duplicates()

population_long = population_long.drop_duplicates()

land_long = land_long.drop_duplicates()

print("Duplicates removed successfully.")

Duplicates removed successfully.


# Phase 8 — Remove Regions & Aggregates

In [58]:
# ==========================================================
# Phase 8.1 - Remove Regions & Aggregates
# ==========================================================

exclude_food = [

    "Africa",
    "Northern Africa",
    "Eastern Africa",
    "Middle Africa",
    "Southern Africa",
    "Western Africa",

    "Americas",
    "Northern America",
    "Central America",
    "Caribbean",
    "South America",

    "Asia",
    "Central Asia",
    "Eastern Asia",
    "Southern Asia",
    "South-eastern Asia",
    "Western Asia",

    "Europe",
    "Eastern Europe",
    "Northern Europe",
    "Southern Europe",
    "Western Europe",

    "Oceania",
    "Australia and New Zealand",
    "Melanesia",
    "Micronesia",
    "Polynesia",

    "European Union (27)",
    "European Union (28)",

    "Least Developed Countries (LDCs)",
    "Land Locked Developing Countries (LLDCs)",
    "Low Income Food Deficit Countries (LIFDCs)",
    "Net Food Importing Developing Countries (NFIDCs)",
    "Small Island Developing States (SIDS)",
    "China",
    "World" 
]

In [59]:
# ==========================================================
# Phase 8.2 - Remove Regions from Food
# ==========================================================

food_filtered = food_filtered[
    ~food_filtered["Area"].isin(exclude_food)
].copy()

print("Countries remaining:", food_filtered["Area"].nunique())

Countries remaining: 178


In [60]:
# ==========================================================
# Phase 8.3 - Remove Regions from Temperature
# ==========================================================

temperature_filtered = temperature_filtered[
    ~temperature_filtered["Country"].isin(exclude_food)
].copy()

print("Countries remaining:", temperature_filtered["Country"].nunique())

Countries remaining: 251


# Phase 9 — Pivot Food Dataset

In [61]:
# ==========================================================
# Phase 9.1 - Pivot Food Balance
# ==========================================================

food_pivot = (

    food_filtered

    .pivot_table(

        index=["Area", "Year", "Item"],

        columns="Element",

        values="Value",

        aggfunc="first"

    )

    .reset_index()

)

food_pivot.head()

Element,Area,Year,Item,Domestic supply quantity,Export quantity,Food supply (kcal/capita/day),Food supply quantity (kg/capita/yr),Import quantity,Production
0,Afghanistan,2010,"Alcohol, Non-Food",0.00,NaN,NaN,NaN,NaN,0.00
1,Afghanistan,2010,Alcoholic Beverages,3.00,NaN,0.15,0.11,3.00,0.00
2,Afghanistan,2010,Animal Products,NaN,NaN,199.39,NaN,NaN,NaN
3,Afghanistan,2010,Animal fats,48.00,NaN,32.41,1.66,3.00,44.00
4,Afghanistan,2010,Apples and products,87.00,1.00,4.22,2.92,10.00,60.00


In [62]:
# ==========================================================
# Phase 9.2 - Rename Columns
# ==========================================================

food_pivot = food_pivot.rename(

    columns={

        "Area": "Country",

        "Domestic supply quantity": "Domestic_Supply",

        "Export quantity": "Exports",

        "Food supply (kcal/capita/day)": "Calories_per_Capita",

        "Food supply quantity (kg/capita/yr)": "Food_kg_per_Capita",

        "Import quantity": "Imports",

        "Production": "Production"

    }

)

food_pivot.head()

Element,Country,Year,Item,Domestic_Supply,Exports,Calories_per_Capita,Food_kg_per_Capita,Imports,Production
0,Afghanistan,2010,"Alcohol, Non-Food",0.00,NaN,NaN,NaN,NaN,0.00
1,Afghanistan,2010,Alcoholic Beverages,3.00,NaN,0.15,0.11,3.00,0.00
2,Afghanistan,2010,Animal Products,NaN,NaN,199.39,NaN,NaN,NaN
3,Afghanistan,2010,Animal fats,48.00,NaN,32.41,1.66,3.00,44.00
4,Afghanistan,2010,Apples and products,87.00,1.00,4.22,2.92,10.00,60.00


In [63]:
# ==========================================================
# Phase 9.2 - Rename Columns
# ==========================================================

food_pivot = food_pivot.rename(

    columns={

        "Area": "Country",

        "Domestic supply quantity": "Domestic_Supply",

        "Export quantity": "Exports",

        "Food supply (kcal/capita/day)": "Calories_per_Capita",

        "Food supply quantity (kg/capita/yr)": "Food_kg_per_Capita",

        "Import quantity": "Imports",

        "Production": "Production"

    }

)

food_pivot.head()

Element,Country,Year,Item,Domestic_Supply,Exports,Calories_per_Capita,Food_kg_per_Capita,Imports,Production
0,Afghanistan,2010,"Alcohol, Non-Food",0.00,NaN,NaN,NaN,NaN,0.00
1,Afghanistan,2010,Alcoholic Beverages,3.00,NaN,0.15,0.11,3.00,0.00
2,Afghanistan,2010,Animal Products,NaN,NaN,199.39,NaN,NaN,NaN
3,Afghanistan,2010,Animal fats,48.00,NaN,32.41,1.66,3.00,44.00
4,Afghanistan,2010,Apples and products,87.00,1.00,4.22,2.92,10.00,60.00


In [64]:
# ==========================================================
# Phase 9.3 - Verify Pivot
# ==========================================================

print(food_pivot.shape)

display(food_pivot.head())

food_pivot.info()

(272570, 9)


Element,Country,Year,Item,Domestic_Supply,Exports,Calories_per_Capita,Food_kg_per_Capita,Imports,Production
0,Afghanistan,2010,"Alcohol, Non-Food",0.00,NaN,NaN,NaN,NaN,0.00
1,Afghanistan,2010,Alcoholic Beverages,3.00,NaN,0.15,0.11,3.00,0.00
2,Afghanistan,2010,Animal Products,NaN,NaN,199.39,NaN,NaN,NaN
3,Afghanistan,2010,Animal fats,48.00,NaN,32.41,1.66,3.00,44.00
4,Afghanistan,2010,Apples and products,87.00,1.00,4.22,2.92,10.00,60.00


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 272570 entries, 0 to 272569
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Country              272570 non-null  object 
 1   Year                 272570 non-null  int64  
 2   Item                 272570 non-null  object 
 3   Domestic_Supply      265337 non-null  float64
 4   Exports              207421 non-null  float64
 5   Calories_per_Capita  247185 non-null  float64
 6   Food_kg_per_Capita   245175 non-null  float64
 7   Imports              255033 non-null  float64
 8   Production           170968 non-null  float64
dtypes: float64(6), int64(1), object(2)
memory usage: 18.7+ MB


# Phase 10 — Country Harmonization

In [65]:
# ==========================================================
# Phase 10.1 - Country Mapping
# ==========================================================

country_mapping = {

    "Bolivia (Plurinational State of)": "Bolivia",

    "China, mainland": "China",

    "China, Hong Kong SAR": "Hong Kong SAR, China",

    "China, Macao SAR": "Macao SAR, China",

    "Congo": "Congo, Rep.",

    "Democratic Republic of the Congo": "Congo, Dem. Rep.",

    "Egypt": "Egypt, Arab Rep.",

    "Iran (Islamic Republic of)": "Iran, Islamic Rep.",

    "Republic of Korea": "Korea, Rep.",

    "Democratic People's Republic of Korea": "Korea, Dem. People's Rep.",

    "Türkiye": "Turkiye",

    "United Republic of Tanzania": "Tanzania",

    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",

    "United States of America": "United States",

    "Venezuela (Bolivarian Republic of)": "Venezuela, RB",

    "Viet Nam": "Vietnam",

    "Côte d'Ivoire": "Cote d'Ivoire",

    "Republic of Moldova": "Moldova",

    "Netherlands (Kingdom of the)": "Netherlands",

    "Bahamas": "Bahamas, The",

    "Gambia": "Gambia, The",

    "Kyrgyzstan": "Kyrgyz Republic",

    "Lao People's Democratic Republic": "Lao PDR",

    "Micronesia (Federated States of)": "Micronesia, Fed. Sts.",

    "Saint Kitts and Nevis": "St. Kitts and Nevis",

    "Saint Lucia": "St. Lucia",

    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",

    "Cape Verde": "Cabo Verde",

    "Swaziland": "Eswatini",

    "Russia": "Russian Federation",
    "Slovakia": "Slovak Republic",
    "Vietnam": "Viet Nam",
    "Yemen": "Yemen, Rep."

}

In [66]:
# ==========================================================
# Phase 10.2 - Apply Country Mapping
# ==========================================================

food_pivot["Country"] = food_pivot["Country"].replace(country_mapping)

temperature_filtered["Country"] = temperature_filtered["Country"].replace(country_mapping)

print("Country names standardized.")

Country names standardized.


In [67]:
# ==========================================================
# Phase 10.3 - Remaining Country Differences
# ==========================================================

food_only = sorted(
    set(food_pivot["Country"]) - set(gdp_long["Country"])
)

print(food_only)

['China, Taiwan Province of', 'Vietnam']


In [68]:
# ==========================================================
# Phase 10.4 - Remove Taiwan (No World Bank Data)
# ==========================================================

food_pivot = food_pivot[
    food_pivot["Country"] != "China, Taiwan Province of"
].copy()

temperature_filtered = temperature_filtered[
    temperature_filtered["Country"] != "China, Taiwan Province of"
].copy()

print("Food Countries:", food_pivot["Country"].nunique())
print("Temperature Countries:", temperature_filtered["Country"].nunique())

Food Countries: 177
Temperature Countries: 251


In [69]:
# ==========================================================
# Phase 10.5 - Final Country Difference Check
# ==========================================================

food_only = sorted(
    set(food_pivot["Country"]) -
    set(gdp_long["Country"])
)

print(food_only)

['Vietnam']


In [70]:
# ==========================================================
# Re-Apply Country Mapping
# ==========================================================

food_pivot["Country"] = food_pivot["Country"].replace(country_mapping)

temperature_filtered["Country"] = temperature_filtered["Country"].replace(country_mapping)

print("Country mapping applied.")

Country mapping applied.


In [71]:
food_only = sorted(
    set(food_pivot["Country"]) -
    set(gdp_long["Country"])
)

print(food_only)

[]


# Phase 11 — Create the Master Dataset

In [72]:
# ==========================================================
# Phase 11.1 - Merge Temperature
# ==========================================================

master_df = food_pivot.merge(
    temperature_filtered,
    on=["Country", "Year"],
    how="left"
)

print(master_df.shape)

master_df.head()

(270961, 10)


,Country,Year,Item,Domestic_Supply,Exports,Calories_per_Capita,Food_kg_per_Capita,Imports,Production,Temperature_Change
0,Afghanistan,2010,"Alcohol, Non-Food",0.00,NaN,NaN,NaN,NaN,0.00,1.66
1,Afghanistan,2010,Alcoholic Beverages,3.00,NaN,0.15,0.11,3.00,0.00,1.66
2,Afghanistan,2010,Animal Products,NaN,NaN,199.39,NaN,NaN,NaN,1.66
3,Afghanistan,2010,Animal fats,48.00,NaN,32.41,1.66,3.00,44.00,1.66
4,Afghanistan,2010,Apples and products,87.00,1.00,4.22,2.92,10.00,60.00,1.66


In [73]:
# ==========================================================
# Phase 11.2 - Merge GDP
# ==========================================================

master_df = master_df.merge(
    gdp_long,
    on=["Country", "Year"],
    how="left"
)

print(master_df.shape)

(270961, 11)


In [74]:
# ==========================================================
# Phase 11.3 - Merge Population
# ==========================================================

master_df = master_df.merge(
    population_long,
    on=["Country", "Year"],
    how="left"
)

print(master_df.shape)

(270961, 12)


In [75]:
# ==========================================================
# Phase 11.4 - Merge Agricultural Land
# ==========================================================

master_df = master_df.merge(
    land_long,
    on=["Country", "Year"],
    how="left"
)

print(master_df.shape)

(270961, 13)


In [76]:
# ==========================================================
# Phase 11.5 - Verify Merge
# ==========================================================

display(master_df.head())

print()

print("Rows :", len(master_df))
print("Columns :", len(master_df.columns))

print()

print(master_df.columns.tolist())

,Country,Year,Item,Domestic_Supply,Exports,Calories_per_Capita,Food_kg_per_Capita,Imports,Production,Temperature_Change,GDP_per_Capita,Population,Agricultural_Land_Percent
0,Afghanistan,2010,"Alcohol, Non-Food",0.00,NaN,NaN,NaN,NaN,0.00,1.66,560.62,28284089.00,58.13
1,Afghanistan,2010,Alcoholic Beverages,3.00,NaN,0.15,0.11,3.00,0.00,1.66,560.62,28284089.00,58.13
2,Afghanistan,2010,Animal Products,NaN,NaN,199.39,NaN,NaN,NaN,1.66,560.62,28284089.00,58.13
3,Afghanistan,2010,Animal fats,48.00,NaN,32.41,1.66,3.00,44.00,1.66,560.62,28284089.00,58.13
4,Afghanistan,2010,Apples and products,87.00,1.00,4.22,2.92,10.00,60.00,1.66,560.62,28284089.00,58.13



Rows : 270961
Columns : 13

['Country', 'Year', 'Item', 'Domestic_Supply', 'Exports', 'Calories_per_Capita', 'Food_kg_per_Capita', 'Imports', 'Production', 'Temperature_Change', 'GDP_per_Capita', 'Population', 'Agricultural_Land_Percent']


In [77]:
# ==========================================================
# Phase 11.6 - Missing Values
# ==========================================================

missing = (
    master_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)

missing["Percent"] = (
    missing["Missing Values"] /
    len(master_df) *
    100
).round(2)

display(missing)

,Missing Values,Percent
Production,101240,37.36
Exports,65035,24.00
Food_kg_per_Capita,27252,10.06
Calories_per_Capita,25248,9.32
Imports,17468,6.45
Domestic_Supply,7191,2.65
Temperature_Change,5879,2.17
Agricultural_Land_Percent,1559,0.58
GDP_per_Capita,1475,0.54
Population,476,0.18


In [78]:
# ==========================================================
# Phase 11.7 - Final Dataset Summary
# ==========================================================

print("=" * 70)
print("MASTER DATASET SUMMARY")
print("=" * 70)

print()

print("Shape:", master_df.shape)

print()

print("Countries :", master_df["Country"].nunique())

print("Years :", master_df["Year"].min(), "-", master_df["Year"].max())

print("Food Items :", master_df["Item"].nunique())

print()

print(master_df.info())

MASTER DATASET SUMMARY

Shape: (270961, 13)

Countries : 177
Years : 2010 - 2023
Food Items : 119

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270961 entries, 0 to 270960
Data columns (total 13 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Country                    270961 non-null  object 
 1   Year                       270961 non-null  int64  
 2   Item                       270961 non-null  object 
 3   Domestic_Supply            263770 non-null  float64
 4   Exports                    205926 non-null  float64
 5   Calories_per_Capita        245713 non-null  float64
 6   Food_kg_per_Capita         243709 non-null  float64
 7   Imports                    253493 non-null  float64
 8   Production                 169721 non-null  float64
 9   Temperature_Change         265082 non-null  float64
 10  GDP_per_Capita             269486 non-null  float64
 11  Population                 270485 non-null  

In [79]:
# ==========================================================
# Phase 11.8 - Verify Merge Integrity
# ==========================================================

print("Food rows   :", len(food_pivot))
print("Merged rows :", len(master_df))

print()

print("Row Count Match :", len(food_pivot) == len(master_df))

print()

duplicates = master_df.duplicated(
    subset=["Country", "Year", "Item"]
).sum()

print("Duplicate Country-Year-Item rows:", duplicates)

Food rows   : 270961
Merged rows : 270961

Row Count Match : True

Duplicate Country-Year-Item rows: 0


In [80]:
# ==========================================================
# Phase 11.9 - Reorder Columns
# ==========================================================

master_df = master_df[
    [
        "Country",
        "Year",
        "Item",

        "Calories_per_Capita",
        "Food_kg_per_Capita",
        "Domestic_Supply",
        "Production",
        "Imports",
        "Exports",

        "Temperature_Change",
        "GDP_per_Capita",
        "Population",
        "Agricultural_Land_Percent"
    ]
]

master_df.head()

,Country,Year,Item,Calories_per_Capita,Food_kg_per_Capita,Domestic_Supply,Production,Imports,Exports,Temperature_Change,GDP_per_Capita,Population,Agricultural_Land_Percent
0,Afghanistan,2010,"Alcohol, Non-Food",NaN,NaN,0.00,0.00,NaN,NaN,1.66,560.62,28284089.00,58.13
1,Afghanistan,2010,Alcoholic Beverages,0.15,0.11,3.00,0.00,3.00,NaN,1.66,560.62,28284089.00,58.13
2,Afghanistan,2010,Animal Products,199.39,NaN,NaN,NaN,NaN,NaN,1.66,560.62,28284089.00,58.13
3,Afghanistan,2010,Animal fats,32.41,1.66,48.00,44.00,3.00,NaN,1.66,560.62,28284089.00,58.13
4,Afghanistan,2010,Apples and products,4.22,2.92,87.00,60.00,10.00,1.00,1.66,560.62,28284089.00,58.13


In [81]:
# ==========================================================
# Phase 11.9b - Add Region (Continent) for Dashboard Filtering
# ==========================================================

import pycountry
import pycountry_convert as pc

def country_to_region(country_name):
    try:
        country = pycountry.countries.search_fuzzy(country_name)[0]
        continent_code = pc.country_alpha2_to_continent_code(country.alpha_2)
        continent_map = {
            'EU': 'Europe', 'AS': 'Asia', 'AF': 'Africa',
            'NA': 'Americas', 'SA': 'Americas', 'OC': 'Oceania'
        }
        return continent_map.get(continent_code, 'Other')
    except Exception:
        return 'Other'

unique_countries = master_df["Country"].unique()
region_lookup = {c: country_to_region(c) for c in unique_countries}
master_df["Region"] = master_df["Country"].map(region_lookup)

unmapped = master_df.loc[master_df["Region"] == "Other", "Country"].unique()
print(f"Unmapped countries ({len(unmapped)}): {list(unmapped)}")

Unmapped countries (18): ['Bahamas, The', 'Hong Kong SAR, China', 'Macao SAR, China', 'Congo, Rep.', "Korea, Dem. People's Rep.", 'Congo, Dem. Rep.', 'Egypt, Arab Rep.', 'Gambia, The', 'Iran, Islamic Rep.', 'Lao PDR', 'Micronesia, Fed. Sts.', 'Korea, Rep.', 'St. Kitts and Nevis', 'St. Lucia', 'St. Vincent and the Grenadines', 'Timor-Leste', 'Venezuela, RB', 'Yemen, Rep.']


In [86]:
# ==========================================================
# Phase 11.9c - Manual Region Overrides (World Bank naming edge cases)
# ==========================================================
region_overrides = {
    "Bahamas, The": "Americas",
    "Hong Kong SAR, China": "Asia",
    "Macao SAR, China": "Asia",
    "Congo, Rep.": "Africa",
    "Congo, Dem. Rep.": "Africa",
    "Korea, Dem. People's Rep.": "Asia",
    "Korea, Rep.": "Asia",
    "Egypt, Arab Rep.": "Africa",
    "Gambia, The": "Africa",
    "Iran, Islamic Rep.": "Asia",
    "Lao PDR": "Asia",
    "Micronesia, Fed. Sts.": "Oceania",
    "St. Kitts and Nevis": "Americas",
    "St. Lucia": "Americas",
    "St. Vincent and the Grenadines": "Americas",
    "Timor-Leste": "Asia",
    "Venezuela, RB": "Americas",
    "Yemen, Rep.": "Asia",
}

for country, region in region_overrides.items():
    master_df.loc[master_df["Country"] == country, "Region"] = region

unmapped_after = master_df.loc[master_df["Region"] == "Other", "Country"].unique()
print(f"Still unmapped ({len(unmapped_after)}): {list(unmapped_after)}")

Still unmapped (0): []


In [90]:
world_check = master_df[master_df["Country"] == "World"]
print("Rows:", len(world_check))   # should be 0

unmapped_final = master_df.loc[master_df["Region"] == "Other", "Country"].unique()
print(f"Unmapped ({len(unmapped_final)}): {list(unmapped_final)}")   # should be empty

Rows: 0
Unmapped (0): []


In [91]:
# ==========================================================
# Phase 11.10 - Final Validation
# ==========================================================

print("=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print()

print("Countries :", master_df["Country"].nunique())
print("Years     :", master_df["Year"].nunique())
print("Items     :", master_df["Item"].nunique())

print()

print(
    "Year Range:",
    master_df["Year"].min(),
    "-",
    master_df["Year"].max()
)

print()

print(
    "Duplicate rows:",
    master_df.duplicated(
        ["Country", "Year", "Item"]
    ).sum()
)

print()

print("Missing Country :", master_df["Country"].isna().sum())
print("Missing Year    :", master_df["Year"].isna().sum())
print("Missing Item    :", master_df["Item"].isna().sum())

FINAL VALIDATION

Countries : 177
Years     : 14
Items     : 119

Year Range: 2010 - 2023

Duplicate rows: 0

Missing Country : 0
Missing Year    : 0
Missing Item    : 0


In [92]:
# ==========================================================
# Phase 11.11 - Save Master Dataset
# ==========================================================

master_df.to_csv(
    PROCESSED_DATA_DIR / "Food_Security_Master_Dataset.csv",
    index=False
)

master_df.to_csv(
    PROCESSED_DATA_DIR / "Food_Security_Master_Dataset.zip",
    index=False,
    compression="zip"
)

print("Food Security Master Dataset saved successfully.")

Food Security Master Dataset saved successfully.
